# Continual-learning experiment Binary Classification


In [4]:
# Cell 1 — imports

from dataclasses import dataclass
from collections.abc import Callable, Iterable
import math
import re
import random

import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

import hdbscan
import umap
from sklearn.metrics import accuracy_score, f1_score
from sklearn.preprocessing import RobustScaler
from transformers import AutoModel, AutoTokenizer

from dataset_utils.constants import Cresci17SetTypes
from dataset_utils.Cresci17 import Cresci17
from dataset_utils.Twibot20 import Twibot20
from dataset_utils.InterleavedIterableDataset import InterleavedIterableDataset

from label_remapper import LabelRemapper
from continual_experiment_manager import ContinualExperimentManager
from ClassificationHeads import ProgressiveNeuralNetworkClassifier
from pipeline_utils import train_classifier


In [5]:
class CalibratedProgressiveClassifier(nn.Module):
    """
    Wraps the PNN with one trainable scale and bias per output class.

    The old PNN heads can remain frozen, while the calibration parameters
    learn how the independently trained head logits should be compared.
    """

    def __init__(
        self,
        base_classifier: nn.Module,
        num_classes: int,
    ):
        super().__init__()

        self.base_classifier = base_classifier

        # exp(0) = 1, so calibration initially leaves logits unchanged.
        self.log_scales = nn.Parameter(
            torch.zeros(num_classes)
        )

        self.biases = nn.Parameter(
            torch.zeros(num_classes)
        )

    @property
    def num_classes(self) -> int:
        return int(self.log_scales.numel())

    def forward(self, features):
        raw_logits = self.base_classifier(features)

        scales = torch.exp(self.log_scales).unsqueeze(0)
        biases = self.biases.unsqueeze(0)

        return raw_logits * scales + biases

    def expand_classifier(self, num_to_add: int):
        """
        Expand both the underlying PNN and its calibration parameters.
        """
        new_head_parameters = list(
            self.base_classifier.expand_classifier(
                num_to_add=num_to_add
            )
        )

        device = self.log_scales.device

        expanded_log_scales = torch.cat(
            [
                self.log_scales.detach(),
                torch.zeros(num_to_add, device=device),
            ]
        )

        expanded_biases = torch.cat(
            [
                self.biases.detach(),
                torch.zeros(num_to_add, device=device),
            ]
        )

        # Register the expanded vectors as new trainable parameters.
        self.log_scales = nn.Parameter(expanded_log_scales)
        self.biases = nn.Parameter(expanded_biases)

        # Train the new PNN head and calibration for all outputs.
        return new_head_parameters + [
            self.log_scales,
            self.biases,
        ]

In [6]:
# Cell 2 — configuration and frozen DistilBERT

DATASET_ROOT = "./datasets"

MAX_TWEETS_PER_USER = 20
TWEET_BATCH_SIZE = 32
MAX_TOKEN_LENGTH = 128

UMAP_COMPONENTS = 15
UMAP_NEIGHBORS = 20

HDBSCAN_MIN_CLUSTER_SIZE = 10
HDBSCAN_CURRENT_FRACTION = 0.80

EPOCHS = 15
LEARNING_RATE = 1e-2

# Ground-truth intervention is useful for measuring continual classification
# independently from novelty-detection errors.
USE_INTERVENTION_OVERRIDE = True

REPLAY_PER_CLASS = 200
EVAL_BATCH_SIZE = 512
RANDOM_SEED = 42

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
bert_model = AutoModel.from_pretrained("distilbert-base-uncased").to(device)

bert_model.eval()
for parameter in bert_model.parameters():
    parameter.requires_grad_(False)

scaler = RobustScaler(with_centering=False)
dim_reducer = umap.UMAP(
    n_components=UMAP_COMPONENTS,
    n_neighbors=UMAP_NEIGHBORS,
    min_dist=0.1,
    metric="cosine",
    random_state=RANDOM_SEED,
)


Device: cuda


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 3536.30it/s]
DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [7]:
# Cell 3 — binary Cresci-2017 task definitions

@dataclass(frozen=True)
class TaskDefinition:
    name: str
    train_factory: Callable[[], Iterable]
    test_factory: Callable[[], Iterable]
    label_transform: Callable[[str], str]


def binary_label_transform(label: str) -> str:
    """
    Converts all Cresci bot subsets into one shared 'bot' label.
    Genuine users become 'human'.
    """
    label = str(label).strip()

    if label == str(Cresci17SetTypes.GENUINE_USER.value):
        return "human"

    return "bot"

BINARY_LABEL_TO_INDEX = {
    "human": 0,
    "bot": 1,
}


def make_binary_cresci_task(
    bot_subset: Cresci17SetTypes,
) -> TaskDefinition:
    """
    Creates one binary task:
        genuine users vs one selected Cresci bot subtype.
    """

    def make_split(mode: str):
        return InterleavedIterableDataset(
            datasets=[
                Cresci17(
                    subset_type=Cresci17SetTypes.GENUINE_USER,
                    mode=mode,
                    root=DATASET_ROOT,
                ),
                Cresci17(
                    subset_type=bot_subset,
                    mode=mode,
                    root=DATASET_ROOT,
                ),
            ],
            mode="RoundRobin",
        )

    return TaskDefinition(
        name=f"Cresci17_Human_vs_{bot_subset.name}",
        train_factory=lambda: make_split("train"),
        test_factory=lambda: make_split("test"),
        label_transform=binary_label_transform,
    )


BINARY_CRESCI_SUBSETS = [
    Cresci17SetTypes.FAKE_FOLLOWER,
    Cresci17SetTypes.SOCIAL_SPAM_1,
    Cresci17SetTypes.TRADITIONAL_SPAM_1,
]


TASK_DEFINITIONS = [
    make_binary_cresci_task(subset)
    for subset in BINARY_CRESCI_SUBSETS
]


for index, task in enumerate(TASK_DEFINITIONS):
    print(index, task.name)

0 Cresci17_Human_vs_FAKE_FOLLOWER
1 Cresci17_Human_vs_SOCIAL_SPAM_1
2 Cresci17_Human_vs_TRADITIONAL_SPAM_1


In [8]:
from collections import Counter
from itertools import islice

for task in TASK_DEFINITIONS:
    labels = [
        task.label_transform(sample.label)
        for sample in islice(task.train_factory(), 30)
    ]

    print(task.name)
    print(Counter(labels))
    print()

Cresci17_Human_vs_FAKE_FOLLOWER
Counter({'human': 15, 'bot': 15})

Cresci17_Human_vs_SOCIAL_SPAM_1
Counter({'human': 15, 'bot': 15})

Cresci17_Human_vs_TRADITIONAL_SPAM_1
Counter({'human': 15, 'bot': 15})



In [9]:
# Cell 4 — embedding helpers for the new Sample dataclass

URL_PATTERN = re.compile(r"https?://\S+|www\.\S+", flags=re.IGNORECASE)


def safe_float(value, default: float = 0.0) -> float:
    if value is None:
        return default

    try:
        result = float(value)
    except (TypeError, ValueError):
        return default

    if not math.isfinite(result):
        return default

    return result


def safe_log1p(value) -> float:
    return math.log1p(max(safe_float(value), 0.0))


def create_profile_vector(user) -> torch.Tensor:
    values = [
        len(str(user.name or "")),
        len(str(user.screen_name or "")),
        safe_log1p(user.statuses_count),
        safe_log1p(user.followers_count),
        safe_log1p(user.friends_count),
        safe_log1p(user.favourites_count),
        0.0,  # protected is unavailable in the current common dataclass
        float(bool(user.verified)),
    ]

    return torch.tensor(values, dtype=torch.float32)


def mean_pool(last_hidden_state, attention_mask):
    mask = attention_mask.unsqueeze(-1).to(last_hidden_state.dtype)
    summed = (last_hidden_state * mask).sum(dim=1)
    counts = mask.sum(dim=1).clamp(min=1.0)
    return summed / counts


def embed_split(
    dataset: Iterable,
    task_name: str,
    label_transform: Callable[[str], str],
    max_samples: int | None = None,
):
    features = []
    labels = []

    print(f"Embedding '{task_name}'...")

    with torch.inference_mode():
        for sample_index, sample in enumerate(dataset):
            if max_samples is not None and sample_index >= max_samples:
                break

            if sample_index % 100 == 0:
                print(f"  {sample_index} users")

            profile_vector = create_profile_vector(sample.user_data)

            texts = []
            for tweet in sample.tweet_data[:MAX_TWEETS_PER_USER]:
                text = str(tweet.text or "").strip()
                if text:
                    texts.append(URL_PATTERN.sub(" url ", text))

            if not texts:
                user_tweet_vector = torch.zeros(
                    bert_model.config.hidden_size,
                    dtype=torch.float32,
                )
            else:
                tweet_batches = []

                for start in range(0, len(texts), TWEET_BATCH_SIZE):
                    batch_texts = texts[start:start + TWEET_BATCH_SIZE]

                    tokens = tokenizer(
                        batch_texts,
                        padding=True,
                        truncation=True,
                        max_length=MAX_TOKEN_LENGTH,
                        return_tensors="pt",
                    )
                    tokens = {
                        name: tensor.to(device)
                        for name, tensor in tokens.items()
                    }

                    outputs = bert_model(**tokens)
                    pooled = mean_pool(
                        outputs.last_hidden_state,
                        tokens["attention_mask"],
                    )
                    tweet_batches.append(pooled.cpu())

                user_tweet_vector = torch.cat(
                    tweet_batches,
                    dim=0,
                ).mean(dim=0)

            combined = torch.cat(
                [profile_vector, user_tweet_vector.float()],
                dim=0,
            )

            features.append(combined.numpy().astype(np.float32))
            labels.append(label_transform(str(sample.label)))

    if not features:
        feature_dim = 8 + bert_model.config.hidden_size
        return (
            np.empty((0, feature_dim), dtype=np.float32),
            [],
        )

    result = np.stack(features).astype(np.float32)
    print(f"Finished '{task_name}': {result.shape}, labels={sorted(set(labels))}")
    return result, labels


In [10]:
# Cell 5 — task-aware binary PNN helpers

class TaskHeadView(nn.Module):
    """
    Selects one binary head from the full expanded PNN output.

    Task 0 -> logits 0 and 1
    Task 1 -> logits 2 and 3
    Task 2 -> logits 4 and 5
    """

    def __init__(
        self,
        classifier: nn.Module,
        task_index: int,
    ):
        super().__init__()

        self.classifier = classifier
        self.task_index = task_index

    def forward(self, features):
        full_logits = self.classifier(features)

        start = 2 * self.task_index
        end = start + 2

        return full_logits[:, start:end]


def binary_labels_to_indices(labels):
    unknown_labels = set(labels) - set(BINARY_LABEL_TO_INDEX)

    if unknown_labels:
        raise ValueError(
            f"Unexpected labels: {unknown_labels}"
        )

    return [
        BINARY_LABEL_TO_INDEX[label]
        for label in labels
    ]


def assert_binary_labels(labels, split_name):
    if set(labels) != {"human", "bot"}:
        raise RuntimeError(
            f"{split_name} must contain both "
            f"'human' and 'bot', but got {set(labels)}"
        )


def transform_features(raw_features):
    scaled = scaler.transform(raw_features)
    reduced = dim_reducer.transform(scaled)

    return np.asarray(
        reduced,
        dtype=np.float32,
    )


def evaluate_binary_head(
    task_head,
    features,
    labels,
):
    dataset = TensorDataset(
        torch.tensor(
            features,
            dtype=torch.float32,
        ),
        torch.tensor(
            labels,
            dtype=torch.long,
        ),
    )

    loader = DataLoader(
        dataset,
        batch_size=EVAL_BATCH_SIZE,
        shuffle=False,
    )

    task_head.eval()

    predictions = []
    ground_truths = []

    with torch.inference_mode():
        for batch_features, batch_labels in loader:
            logits = task_head(
                batch_features.to(device)
            )

            batch_predictions = (
                logits.argmax(dim=1)
                .cpu()
                .numpy()
            )

            predictions.extend(
                batch_predictions.tolist()
            )

            ground_truths.extend(
                batch_labels.numpy().tolist()
            )

    return ground_truths, predictions

In [11]:
# Cell 6 — experiment objects

experiment_manager = ContinualExperimentManager(
    num_tasks=len(TASK_DEFINITIONS),
    use_intervention_override=True,
)

criterion = nn.CrossEntropyLoss()

classifier = None

# Stores:
# (test features, binary labels, task name, task-head index)
test_splits_cache = []

In [12]:
# Cell 7 — binary task-aware Cresci continual-learning loop

for task_index, task in enumerate(TASK_DEFINITIONS):
    print("\n" + "=" * 72)
    print(f"STEP {task_index}: {task.name}")
    print("=" * 72)

    # ---------------------------------------------------------
    # 1. Embed current binary training task
    # ---------------------------------------------------------
    raw_train, train_labels = embed_split(
        dataset=task.train_factory(),
        task_name=f"{task.name}/train",
        label_transform=task.label_transform,
    )

    train_labels = [
        str(label)
        for label in train_labels
    ]

    assert_binary_labels(
        train_labels,
        split_name=f"{task.name}/train",
    )

    train_targets = binary_labels_to_indices(
        train_labels
    )

    print(
        "Training-label counts:",
        {
            label: train_labels.count(label)
            for label in ["human", "bot"]
        },
    )

    # ---------------------------------------------------------
    # 2. Fit UMAP only on Task 0
    # ---------------------------------------------------------
    if task_index == 0:
        scaled_train = scaler.fit_transform(
            raw_train
        )

        umap_train = dim_reducer.fit_transform(
            scaled_train
        )

        umap_train = np.asarray(
            umap_train,
            dtype=np.float32,
        )

    else:
        umap_train = transform_features(
            raw_train
        )

    # ---------------------------------------------------------
    # 3. Create or expand one binary PNN head
    # ---------------------------------------------------------
    if task_index == 0:
        base_classifier = (
            ProgressiveNeuralNetworkClassifier(
                in_features=UMAP_COMPONENTS,
                output_dim=2,
                dropout_p=0.1,
            )
        )

        classifier = CalibratedProgressiveClassifier(
            base_classifier=base_classifier,
            num_classes=2,
        ).to(device)

        trainable_parameters = list(
            classifier.parameters()
        )

        print(
            "Created Task 0 binary head: "
            "[human, bot]"
        )

    else:
        # Add TWO logits for each new binary task.
        trainable_parameters = list(
            classifier.expand_classifier(
                num_to_add=2
            )
        )

        print(
            f"Added Task {task_index} binary head: "
            "[human, bot]"
        )

    # ---------------------------------------------------------
    # 4. Select only the current task's two logits
    # ---------------------------------------------------------
    current_task_head = TaskHeadView(
        classifier=classifier,
        task_index=task_index,
    ).to(device)

    classifier.eval()

    with torch.inference_mode():
        probe = torch.tensor(
            umap_train[:2],
            dtype=torch.float32,
            device=device,
        )

        all_outputs = classifier(probe).shape[1]
        current_outputs = (
            current_task_head(probe)
            .shape[1]
        )

    expected_outputs = 2 * (task_index + 1)

    if all_outputs != expected_outputs:
        raise RuntimeError(
            f"Expected {expected_outputs} total outputs, "
            f"but got {all_outputs}."
        )

    if current_outputs != 2:
        raise RuntimeError(
            f"Expected 2 outputs for current task, "
            f"but got {current_outputs}."
        )

    print(
        f"Total PNN outputs: {all_outputs}"
    )

    print(
        "Current task-head outputs: 2"
    )

    # ---------------------------------------------------------
    # 5. Train only the current binary task head
    # ---------------------------------------------------------
    optimizer = torch.optim.Adam(
        trainable_parameters,
        lr=LEARNING_RATE,
    )

    current_task_head.train()

    train_classifier(
        current_task_head,
        criterion,
        optimizer,
        umap_train,
        train_targets,
        epochs=EPOCHS,
        device=device,
    )

    # ---------------------------------------------------------
    # 6. Embed and cache the current binary test task
    # ---------------------------------------------------------
    raw_test, test_labels = embed_split(
        dataset=task.test_factory(),
        task_name=f"{task.name}/test",
        label_transform=task.label_transform,
    )

    test_labels = [
        str(label)
        for label in test_labels
    ]

    assert_binary_labels(
        test_labels,
        split_name=f"{task.name}/test",
    )

    test_targets = binary_labels_to_indices(
        test_labels
    )

    umap_test = transform_features(
        raw_test
    )

    test_splits_cache.append(
        (
            umap_test,
            test_targets,
            task.name,
            task_index,
        )
    )

    # ---------------------------------------------------------
    # 7. Evaluate every old task with ITS OWN binary head
    # ---------------------------------------------------------
    print(
        f"Evaluating {len(test_splits_cache)} "
        "binary task(s)..."
    )

    for evaluated_task_index, (
        cached_features,
        cached_labels,
        cached_name,
        head_index,
    ) in enumerate(test_splits_cache):

        evaluation_head = TaskHeadView(
            classifier=classifier,
            task_index=head_index,
        ).to(device)

        ground_truths, predictions = (
            evaluate_binary_head(
                task_head=evaluation_head,
                features=cached_features,
                labels=cached_labels,
            )
        )

        accuracy = accuracy_score(
            ground_truths,
            predictions,
        )

        macro_f1 = f1_score(
            ground_truths,
            predictions,
            average="macro",
            zero_division=0,
        )

        print(
            f"  Task {evaluated_task_index} "
            f"— {cached_name}: "
            f"accuracy={accuracy:.4f}, "
            f"macro-F1={macro_f1:.4f}"
        )

        experiment_manager.record_accuracy(
            task_index=evaluated_task_index,
            accuracy=accuracy,
        )

    experiment_manager.advance_to_next_task()


experiment_manager.summary()


STEP 0: Cresci17_Human_vs_FAKE_FOLLOWER
Embedding 'Cresci17_Human_vs_FAKE_FOLLOWER/train'...
  0 users
  100 users
  200 users
  300 users
  400 users
  500 users
  600 users
  700 users
  800 users
  900 users
  1000 users
  1100 users
  1200 users
  1300 users
  1400 users
  1500 users
  1600 users
  1700 users
  1800 users
  1900 users
  2000 users
  2100 users
  2200 users
  2300 users
  2400 users
  2500 users
  2600 users
  2700 users
  2800 users
  2900 users
  3000 users
  3100 users
  3200 users
  3300 users
  3400 users
  3500 users
  3600 users
  3700 users
  3800 users
  3900 users
  4000 users
  4100 users
  4200 users
  4300 users
  4400 users
  4500 users
  4600 users
  4700 users
  4800 users
  4900 users
Finished 'Cresci17_Human_vs_FAKE_FOLLOWER/train': (4911, 776), labels=['bot', 'human']
Training-label counts: {'human': 2563, 'bot': 2348}


c:\Users\Hamouda\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


Created Task 0 binary head: [human, bot]
Total PNN outputs: 2
Current task-head outputs: 2
Epoch [1/15] - Loss: 0.8042
Epoch [2/15] - Loss: 0.2617
Epoch [3/15] - Loss: 0.2423
Epoch [4/15] - Loss: 0.2435
Epoch [5/15] - Loss: 0.2342
Epoch [6/15] - Loss: 0.2308
Epoch [7/15] - Loss: 0.2347
Epoch [8/15] - Loss: 0.2284
Epoch [9/15] - Loss: 0.2343
Epoch [10/15] - Loss: 0.2295
Epoch [11/15] - Loss: 0.2286
Epoch [12/15] - Loss: 0.2370
Epoch [13/15] - Loss: 0.2391
Epoch [14/15] - Loss: 0.2355
Epoch [15/15] - Loss: 0.2348
Embedding 'Cresci17_Human_vs_FAKE_FOLLOWER/test'...
  0 users
  100 users
  200 users
  300 users
  400 users
  500 users
  600 users
Finished 'Cresci17_Human_vs_FAKE_FOLLOWER/test': (604, 776), labels=['bot', 'human']
Evaluating 1 binary task(s)...
  Task 0 — Cresci17_Human_vs_FAKE_FOLLOWER: accuracy=0.9139, macro-F1=0.9135

--- Step 0 complete ---
    Avg Accuracy (this step): 0.9139
    Forgetting Measure:       0.0000
    Row scores: ['0.914']

=== Now starting Step 1 ===




In [13]:
print(
    "Label mapping:",
    {
        label: label_remapper.convert([label])[0]
        for label in sorted(known_native_labels)
    },
)

print("Known labels:", sorted(known_native_labels))

print(
    "Replay classes:",
    {
        label: len(features)
        for label, features in replay_by_label.items()
    },
)

NameError: name 'known_native_labels' is not defined

## Diagnostic

In [ ]:
from collections import Counter

import numpy as np
import torch

from sklearn.metrics import confusion_matrix, classification_report


# Reconstruct label mapping.
label_to_index = {
    label: label_remapper.convert([label])[0]
    for label in sorted(known_native_labels)
}

index_to_label = {
    index: label
    for label, index in label_to_index.items()
}

ordered_indices = sorted(index_to_label)

print("Label mapping:")
print(label_to_index)


all_ground_truths = []
all_predictions = []

for task_index, (
    cached_features,
    cached_labels,
    cached_name,
) in enumerate(test_splits_cache):

    ground_truths, predictions = evaluate_classifier(
        classifier,
        cached_features,
        cached_labels,
    )

    readable_truths = [
        index_to_label[index]
        for index in ground_truths
    ]

    readable_predictions = [
        index_to_label[index]
        for index in predictions
    ]

    print("\n" + "=" * 75)
    print(f"TASK {task_index}: {cached_name}")

    print("\nTrue-label counts:")
    print(Counter(readable_truths))

    print("\nPredicted-label counts:")
    print(Counter(readable_predictions))

    print("\nConfusion matrix:")
    print(
        confusion_matrix(
            ground_truths,
            predictions,
            labels=ordered_indices,
        )
    )

    all_ground_truths.extend(ground_truths)
    all_predictions.extend(predictions)


print("\n" + "=" * 75)
print("COMBINED TEST-SET REPORT")

print(
    classification_report(
        all_ground_truths,
        all_predictions,
        labels=ordered_indices,
        target_names=[
            index_to_label[index]
            for index in ordered_indices
        ],
        zero_division=0,
    )
)

print("Combined confusion matrix:")

print(
    confusion_matrix(
        all_ground_truths,
        all_predictions,
        labels=ordered_indices,
    )
)

Label mapping:
{'fake_followers': 0, 'genuine_user': 1, 'social_spambots_1': 2, 'traditional_spambots_1': 3}

TASK 0: Cresci17_Initial_Genuine_vs_FakeFollower

True-label counts:
Counter({'genuine_user': 323, 'fake_followers': 281})

Predicted-label counts:
Counter({'genuine_user': 255, 'fake_followers': 199, 'traditional_spambots_1': 88, 'social_spambots_1': 62})

Confusion matrix:
[[195  10  57  19]
 [  4 245   5  69]
 [  0   0   0   0]
 [  0   0   0   0]]

TASK 1: Cresci17_SOCIAL_SPAM_1

True-label counts:
Counter({'social_spambots_1': 13})

Predicted-label counts:
Counter({'social_spambots_1': 12, 'traditional_spambots_1': 1})

Confusion matrix:
[[ 0  0  0  0]
 [ 0  0  0  0]
 [ 0  0 12  1]
 [ 0  0  0  0]]

TASK 2: Cresci17_TRADITIONAL_SPAM_1

True-label counts:
Counter({'traditional_spambots_1': 110})

Predicted-label counts:
Counter({'traditional_spambots_1': 50, 'social_spambots_1': 34, 'genuine_user': 13, 'fake_followers': 13})

Confusion matrix:
[[ 0  0  0  0]
 [ 0  0  0  0]
 [

In [ ]:
for task_index, (
    cached_features,
    cached_labels,
    cached_name,
) in enumerate(test_splits_cache):

    features_tensor = torch.tensor(
        cached_features,
        dtype=torch.float32,
        device=device,
    )

    classifier.eval()

    with torch.inference_mode():
        logits = classifier(features_tensor).cpu().numpy()

    mean_logits = logits.mean(axis=0)
    prediction_counts = np.bincount(
        logits.argmax(axis=1),
        minlength=len(index_to_label),
    )

    print("\n" + "=" * 75)
    print(cached_name)

    print("Average logit from each output:")
    for class_index, mean_logit in enumerate(mean_logits):
        print(
            f"  {class_index} "
            f"({index_to_label[class_index]}): "
            f"{mean_logit:.4f}"
        )

    print("Prediction count from each output:")
    for class_index, count in enumerate(prediction_counts):
        print(
            f"  {class_index} "
            f"({index_to_label[class_index]}): "
            f"{count}"
        )


Cresci17_Initial_Genuine_vs_FakeFollower
Average logit from each output:
  0 (fake_followers): 0.8795
  1 (genuine_user): 0.8792
  2 (social_spambots_1): -1.7068
  3 (traditional_spambots_1): 0.4252
Prediction count from each output:
  0 (fake_followers): 199
  1 (genuine_user): 255
  2 (social_spambots_1): 62
  3 (traditional_spambots_1): 88

Cresci17_SOCIAL_SPAM_1
Average logit from each output:
  0 (fake_followers): 3.3811
  1 (genuine_user): 0.2247
  2 (social_spambots_1): 3.7045
  3 (traditional_spambots_1): 2.4693
Prediction count from each output:
  0 (fake_followers): 0
  1 (genuine_user): 0
  2 (social_spambots_1): 12
  3 (traditional_spambots_1): 1

Cresci17_TRADITIONAL_SPAM_1
Average logit from each output:
  0 (fake_followers): 1.8645
  1 (genuine_user): 0.6551
  2 (social_spambots_1): 0.2099
  3 (traditional_spambots_1): 1.7402
Prediction count from each output:
  0 (fake_followers): 13
  1 (genuine_user): 13
  2 (social_spambots_1): 34
  3 (traditional_spambots_1): 50
